# Tomahawk Cassette Rev3.1 — corrected deposition and full-incident validation screen

This notebook replaces the earlier mixed heat-load interpretation with three explicitly separated calculations:

1. **Transport-deposition controlled envelope:** 0.650–0.900 MW/m² local nuclear heating, centered on the earlier 0.772 MW/m² multi-layer screening estimate. These are pre-transport screening inputs and must be replaced by OpenMC/MCNP material-heating tallies.
2. **Full-incident engineered envelope:** the complete 4 MW/m² wall load is applied as cassette heat across six flow-matched hydraulic layouts and declared operating/material ranges.
3. **Full-incident broad challenge:** the same 4 MW/m² load is applied to a deliberately broad Cartesian grid, including unsuitable combinations, to preserve the failure map.

The notebook reports branch-specific pass fractions. They are deterministic coverage of the declared input grids—not probabilities of hardware success. Full-incident passing rows require very high water mass flow; no plant-level mass-flow feasibility gate has been established. It uses NumPy, Pandas, Matplotlib, and Python's standard library only; no GPU is required.

In [1]:
# SECTION 1 — imports, deterministic configuration, and output paths
import hashlib
import json
import math
import platform
import time
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

SEED = 20260826
np.random.seed(SEED)
OUT_DIR = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd() / 'tomahawk_cassette_rev3_outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROLLED_NAME = 'Tomahawk_Cassette_Rev3_Controlled_Envelope_Input.csv'
ENGINEERED_NAME = 'Tomahawk_Cassette_Rev3_Engineered_Full_Incident_Input.csv'
STRESS_NAME = 'Tomahawk_Cassette_Rev3_Full_Incident_Stress_Input.csv'

CONSTANTS = {
    'n_cassettes_per_module': 4,
    'cassette_area_m2': 2.5,
    'rf_power_absorbed_MW_module': 15.0,
    'rho_kg_m3': 950.0,
    'mu_Pa_s': 2.30e-4,
    'water_k_W_mK': 0.68,
    'cp_J_kgK': 4180.0,
    'cucrzr_k_W_mK': 300.0,
    'minor_loss_K': 3.0,
    'pump_efficiency': 0.70,
}

GATES = {
    'max_CuCrZr_temperature_C': 300.0,
    'minimum_boiling_margin_C': 10.0,
    'max_velocity_m_s': 5.0,
    'max_pressure_drop_MPa': 0.25,
    'max_pump_power_MW_module': 1.0,
    'max_flow_area_fraction': 0.08,
}

TOPOLOGY = {
    'parallel_smooth': {'Nu_multiplier': 1.00, 'dp_multiplier': 1.00},
    'parallel_ribbed': {'Nu_multiplier': 1.25, 'dp_multiplier': 1.60},
}

print('Python:', platform.python_version())
print('Output:', OUT_DIR)
print('Seed:', SEED)
print('GPU required: no')

Python: 3.12.13
Output: /workspace/scratch/9740310843df/tomahawk_cassette_rev3_outputs
Seed: 20260826
GPU required: no


In [2]:
# SECTION 2 — locate and hash the two external input datasets
def find_dataset(filename):
    candidates = [Path.cwd() / filename]
    if Path('/kaggle/input').exists():
        candidates.extend(Path('/kaggle/input').rglob(filename))
    for path in candidates:
        if path.exists():
            return path.resolve()
    raise FileNotFoundError(f'Upload {filename} as a Kaggle dataset before running.')

controlled_path = find_dataset(CONTROLLED_NAME)
engineered_path = find_dataset(ENGINEERED_NAME)
stress_path = find_dataset(STRESS_NAME)
controlled_input = pd.read_csv(controlled_path)
engineered_input = pd.read_csv(engineered_path)
stress_input = pd.read_csv(stress_path)
inputs = pd.concat([controlled_input, engineered_input, stress_input], ignore_index=True)

required = [
    'analysis_branch', 'topology', 'q_nuclear_areal_MW_m2', 'rf_loss_fraction',
    'pressure_MPa', 'inlet_temp_C', 'deltaT_C', 'channel_count_per_cassette',
    'Dh_mm', 'cucrzr_skin_mm', 'channel_length_m'
]
missing = [c for c in required if c not in inputs]
if missing:
    raise ValueError(f'Missing required columns: {missing}')
if not np.isfinite(inputs[required[2:]].to_numpy(dtype=float)).all():
    raise ValueError('Non-finite numeric input found.')
if not set(inputs['topology']).issubset(TOPOLOGY):
    raise ValueError('Unknown topology found in input data.')

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

print('Controlled input:', controlled_path, 'rows:', len(controlled_input), 'sha256:', sha256(controlled_path))
print('Engineered full-incident input:', engineered_path, 'rows:', len(engineered_input), 'sha256:', sha256(engineered_path))
print('Stress input:', stress_path, 'rows:', len(stress_input), 'sha256:', sha256(stress_path))
print('Input validation: PASS')

Controlled input: /workspace/scratch/9740310843df/Tomahawk_Cassette_Rev3_Controlled_Envelope_Input.csv rows: 1728 sha256: ad7edd9e93eb8a1e3d6d6d5f3897e952b47a60b6cadd82355b19c126ec03128f
Engineered full-incident input: /workspace/scratch/9740310843df/Tomahawk_Cassette_Rev3_Engineered_Full_Incident_Input.csv rows: 432 sha256: 4a40a8d14440b85222b073906409a339a3bfd03633778f126917c5eefbba6d33
Stress input: /workspace/scratch/9740310843df/Tomahawk_Cassette_Rev3_Full_Incident_Stress_Input.csv rows: 10935 sha256: 60cbec591ff27b1bed3f009bb165fcb515b3b9235200dcf4981b55c0f3683a5e
Input validation: PASS


In [3]:
# SECTION 3 — vectorized hydraulic and thermal evaluator
def saturation_temperature_C(p_MPa):
    return np.interp(np.asarray(p_MPa, dtype=float), [3, 4, 5, 6, 7], [233.9, 250.4, 263.9, 275.6, 285.8])

def evaluate(frame):
    x = frame.copy()
    c = CONSTANTS
    total_area = c['n_cassettes_per_module'] * c['cassette_area_m2']
    q_nuclear_W = x['q_nuclear_areal_MW_m2'].to_numpy(float) * 1e6 * total_area
    q_rf_W = c['rf_power_absorbed_MW_module'] * 1e6 * x['rf_loss_fraction'].to_numpy(float)
    q_total_W = q_nuclear_W + q_rf_W
    dT = x['deltaT_C'].to_numpy(float)
    mdot_module = q_total_W / (c['cp_J_kgK'] * dT)
    mdot_cassette = mdot_module / c['n_cassettes_per_module']

    Dh = x['Dh_mm'].to_numpy(float) / 1000.0
    n_channels = x['channel_count_per_cassette'].to_numpy(float)
    flow_area = n_channels * Dh ** 2
    velocity = mdot_cassette / (c['rho_kg_m3'] * flow_area)
    Re = c['rho_kg_m3'] * velocity * Dh / c['mu_Pa_s']
    Pr = c['cp_J_kgK'] * c['mu_Pa_s'] / c['water_k_W_mK']
    Nu_base = np.where(Re < 2300.0, 4.36, 0.023 * Re ** 0.8 * Pr ** 0.4)
    nu_mult = x['topology'].map({k:v['Nu_multiplier'] for k,v in TOPOLOGY.items()}).to_numpy(float)
    dp_mult = x['topology'].map({k:v['dp_multiplier'] for k,v in TOPOLOGY.items()}).to_numpy(float)
    Nu = Nu_base * nu_mult
    h = Nu * c['water_k_W_mK'] / Dh

    friction_base = np.where(Re < 2300.0, 64.0 / np.maximum(Re, 1.0), 0.3164 / Re ** 0.25)
    dynamic_pressure = 0.5 * c['rho_kg_m3'] * velocity ** 2
    length = x['channel_length_m'].to_numpy(float)
    dp_Pa = (friction_base * dp_mult * length / Dh + c['minor_loss_K'] * dp_mult) * dynamic_pressure

    q_face_W_m2 = q_total_W / total_area
    skin = x['cucrzr_skin_mm'].to_numpy(float) / 1000.0
    wall_temp = (
        x['inlet_temp_C'].to_numpy(float) + dT
        + q_face_W_m2 / h
        + q_face_W_m2 * skin / c['cucrzr_k_W_mK']
    )
    Tsat = saturation_temperature_C(x['pressure_MPa'])
    pump_power_W = dp_Pa * (mdot_module / c['rho_kg_m3']) / c['pump_efficiency']
    flow_area_fraction = flow_area / c['cassette_area_m2']

    x['q_total_MW_module'] = q_total_W / 1e6
    x['mdot_kg_s_module'] = mdot_module
    x['flow_area_m2_cassette'] = flow_area
    x['flow_area_fraction'] = flow_area_fraction
    x['velocity_m_s'] = velocity
    x['Re'] = Re
    x['Nu'] = Nu
    x['h_W_m2K'] = h
    x['pressure_drop_kPa'] = dp_Pa / 1000.0
    x['pump_power_MW_module'] = pump_power_W / 1e6
    x['wall_temp_C'] = wall_temp
    x['saturation_temp_C'] = Tsat
    x['boiling_margin_C'] = Tsat - wall_temp
    x['thermal_margin_C'] = GATES['max_CuCrZr_temperature_C'] - wall_temp

    x['gate_temperature'] = wall_temp <= GATES['max_CuCrZr_temperature_C']
    x['gate_boiling_margin'] = (Tsat - wall_temp) >= GATES['minimum_boiling_margin_C']
    x['gate_velocity'] = velocity <= GATES['max_velocity_m_s']
    x['gate_pressure_drop'] = dp_Pa <= GATES['max_pressure_drop_MPa'] * 1e6
    x['gate_pump_power'] = pump_power_W <= GATES['max_pump_power_MW_module'] * 1e6
    x['gate_flow_area'] = flow_area_fraction <= GATES['max_flow_area_fraction']
    gate_cols = [c for c in x if c.startswith('gate_')]
    x['passes_all_gates'] = x[gate_cols].all(axis=1)
    return x

start = time.perf_counter()
results = evaluate(inputs)
elapsed = time.perf_counter() - start
print('Evaluated rows:', len(results), 'elapsed_seconds:', round(elapsed, 4))

Evaluated rows: 13095 elapsed_seconds: 0.0068


In [4]:
# SECTION 4 — branch-separated results and acceptance checks
gate_cols = [c for c in results if c.startswith('gate_')]
summary = results.groupby('analysis_branch', as_index=False).agg(
    rows=('passes_all_gates', 'size'),
    passing_rows=('passes_all_gates', 'sum'),
    minimum_wall_temp_C=('wall_temp_C', 'min'),
    maximum_wall_temp_C=('wall_temp_C', 'max'),
    minimum_boiling_margin_C=('boiling_margin_C', 'min'),
    maximum_velocity_m_s=('velocity_m_s', 'max'),
    maximum_pressure_drop_kPa=('pressure_drop_kPa', 'max'),
    maximum_pump_power_MW=('pump_power_MW_module', 'max'),
)
summary['pass_percent'] = 100.0 * summary['passing_rows'] / summary['rows']
passing_flow = results[results['passes_all_gates']].groupby('analysis_branch', as_index=False).agg(
    minimum_passing_mass_flow_kg_s_module=('mdot_kg_s_module', 'min'),
    maximum_passing_mass_flow_kg_s_module=('mdot_kg_s_module', 'max'),
)
summary = summary.merge(passing_flow, on='analysis_branch', how='left')
display(summary.round(4))

gate_summary = results.groupby('analysis_branch')[gate_cols].mean().mul(100.0).round(2)
display(gate_summary)

controlled_summary = summary[summary['analysis_branch'].eq('transport_deposition_controlled')].iloc[0]
engineered_summary = summary[summary['analysis_branch'].eq('full_incident_engineered')].iloc[0]
stress_summary = summary[summary['analysis_branch'].eq('full_incident_broad_challenge')].iloc[0]
controlled_gate = bool(controlled_summary['pass_percent'] >= 60.0)
engineered_gate = bool(engineered_summary['pass_percent'] >= 60.0)
stress_nonzero_gate = bool(stress_summary['passing_rows'] > 0)

print('CONTROLLED ENVELOPE >=60% GATE:', 'PASS' if controlled_gate else 'FAIL')
print('ENGINEERED FULL-INCIDENT >=60% GATE:', 'PASS' if engineered_gate else 'FAIL')
print('BROAD FULL-INCIDENT NONZERO SCREEN GATE:', 'PASS' if stress_nonzero_gate else 'FAIL')
print('Interpretation: coverage of declared deterministic grids only; not hardware-success probabilities.')
print('System-integration caution: the minimum passing engineered full-incident flow is',
      round(float(engineered_summary['minimum_passing_mass_flow_kg_s_module']), 2),
      'kg/s/module; no plant-level mass-flow feasibility gate has been established.')

if not controlled_gate:
    raise RuntimeError('Controlled design envelope did not meet the declared 60% screening gate.')
if not engineered_gate:
    raise RuntimeError('Engineered full-incident envelope did not meet the declared 60% screening gate.')
if not stress_nonzero_gate:
    raise RuntimeError('Broad full-incident challenge returned zero passing candidates.')

                analysis_branch  rows  passing_rows  minimum_wall_temp_C  maximum_wall_temp_C  minimum_boiling_margin_C  maximum_velocity_m_s  maximum_pressure_drop_kPa  maximum_pump_power_MW  pass_percent  minimum_passing_mass_flow_kg_s_module  maximum_passing_mass_flow_kg_s_module
  full_incident_broad_challenge 10935           672             156.5440             664.6630                 -400.7630               15.6584                   674.5580                 0.9889        6.1454                               482.0574                               974.8804
       full_incident_engineered   432           336             228.1050             284.4704                   -8.8704                4.9738                    68.1524                 0.0992       77.7778                               482.0574                               967.7033
transport_deposition_controlled  1728          1346             157.8247             347.4344                  -83.5344                2.4208       

In [5]:
# SECTION 5 — rank candidates, plot, and export a complete result bundle
feasible = results[results['passes_all_gates']].copy()
feasible['ranking_score'] = (
    feasible['thermal_margin_C']
    + 0.20 * feasible['boiling_margin_C']
    - 0.08 * feasible['pressure_drop_kPa']
    - 5.0 * feasible['pump_power_MW_module']
    - 2.0 * np.maximum(feasible['velocity_m_s'] - 3.0, 0.0)
    - 0.05 * feasible['mdot_kg_s_module']
)

controlled_top = feasible[feasible['analysis_branch'].eq('transport_deposition_controlled')].sort_values('ranking_score', ascending=False).head(20)
engineered_top = feasible[feasible['analysis_branch'].eq('full_incident_engineered')].sort_values('ranking_score', ascending=False).head(20)
stress_top = feasible[feasible['analysis_branch'].eq('full_incident_broad_challenge')].sort_values('ranking_score', ascending=False).head(20)

print('Top controlled-deposition candidates:')
display(controlled_top[[
    'topology','q_nuclear_areal_MW_m2','rf_loss_fraction','pressure_MPa','inlet_temp_C','deltaT_C',
    'channel_count_per_cassette','Dh_mm','cucrzr_skin_mm','channel_length_m','mdot_kg_s_module',
    'velocity_m_s','pressure_drop_kPa','pump_power_MW_module','wall_temp_C','boiling_margin_C','ranking_score'
]].round(4))

print('Top engineered full-incident candidates:')
display(engineered_top[[
    'topology','q_nuclear_areal_MW_m2','rf_loss_fraction','pressure_MPa','inlet_temp_C','deltaT_C',
    'channel_count_per_cassette','Dh_mm','cucrzr_skin_mm','channel_length_m','mdot_kg_s_module',
    'velocity_m_s','pressure_drop_kPa','pump_power_MW_module','wall_temp_C','boiling_margin_C','ranking_score'
]].round(4))

print('Top broad full-incident challenge candidates:')
display(stress_top[[
    'topology','q_nuclear_areal_MW_m2','rf_loss_fraction','pressure_MPa','inlet_temp_C','deltaT_C',
    'channel_count_per_cassette','Dh_mm','cucrzr_skin_mm','channel_length_m','mdot_kg_s_module',
    'velocity_m_s','pressure_drop_kPa','pump_power_MW_module','wall_temp_C','boiling_margin_C','ranking_score'
]].round(4))

results_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_All_Results.csv'
summary_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Branch_Summary.csv'
controlled_top_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Top_Controlled_Candidates.csv'
engineered_top_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Top_Engineered_Full_Incident_Candidates.csv'
stress_top_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Top_Full_Incident_Candidates.csv'
plot_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Branch_Results.png'
manifest_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Run_Manifest.json'
zip_path = OUT_DIR / 'Tomahawk_Cassette_Rev3_Result_Bundle.zip'

results.to_csv(results_path, index=False)
summary.to_csv(summary_path, index=False)
controlled_top.to_csv(controlled_top_path, index=False)
engineered_top.to_csv(engineered_top_path, index=False)
stress_top.to_csv(stress_top_path, index=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for branch, group in results.groupby('analysis_branch'):
    axes[0].scatter(group['pressure_drop_kPa'], group['wall_temp_C'], s=5, alpha=.2, label=branch)
axes[0].axhline(GATES['max_CuCrZr_temperature_C'], color='red', linestyle='--')
axes[0].set(xlabel='Pressure drop [kPa]', ylabel='Wall temperature [°C]', title='Thermal-hydraulic design space')
axes[0].legend(fontsize=7); axes[0].grid(alpha=.2)
axes[1].bar(summary['analysis_branch'], summary['pass_percent'], color=['#2E74B5','#7A5A00'])
axes[1].axhline(60.0, color='red', linestyle='--', label='controlled-envelope gate')
axes[1].set(ylabel='Passing rows [%]', title='Branch-separated deterministic coverage', ylim=(0,100))
axes[1].tick_params(axis='x', rotation=15); axes[1].legend(fontsize=7); axes[1].grid(axis='y', alpha=.2)
plt.tight_layout(); plt.savefig(plot_path, dpi=170); plt.close()

manifest = {
    'model': 'Tomahawk Cassette Rev3.1 corrected deposition, engineered envelope, and broad challenge screen',
    'seed': SEED,
    'scope': 'reduced-order deterministic thermal-hydraulic screen; not CFD, neutronics, structural, irradiation, or hardware validation',
    'controlled_deposition_basis': '0.650-0.900 MW/m2 screening range centered on prior 0.772 MW/m2 layer estimate; replace with transport tallies',
    'full_incident_engineered_basis': '4 MW/m2 applied as deposited cassette heat across six flow-matched channel layouts and declared engineering ranges',
    'full_incident_stress_basis': '4 MW/m2 applied as deposited cassette heat across a deliberately broad Cartesian challenge grid',
    'system_integration_caution': 'Passing full-incident rows require at least the branch-reported minimum water mass flow; no plant-level mass-flow feasibility gate has been established.',
    'input_hashes': {CONTROLLED_NAME: sha256(controlled_path), ENGINEERED_NAME: sha256(engineered_path), STRESS_NAME: sha256(stress_path)},
    'constants': CONSTANTS,
    'gates': GATES,
    'summary': summary.to_dict(orient='records'),
    'controlled_60_percent_gate': controlled_gate,
    'engineered_full_incident_60_percent_gate': engineered_gate,
    'full_incident_nonzero_gate': stress_nonzero_gate,
    'elapsed_seconds': elapsed,
}
manifest_path.write_text(json.dumps(manifest, indent=2))

files = [results_path, summary_path, controlled_top_path, engineered_top_path, stress_top_path, plot_path, manifest_path]
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for path in files:
        z.write(path, path.name)

print('Saved result bundle:', zip_path)
print(json.dumps(manifest, indent=2))

Top controlled-deposition candidates:
       topology  q_nuclear_areal_MW_m2  rf_loss_fraction  pressure_MPa  inlet_temp_C  deltaT_C  channel_count_per_cassette  Dh_mm  cucrzr_skin_mm  channel_length_m  mdot_kg_s_module  velocity_m_s  pressure_drop_kPa  pump_power_MW_module  wall_temp_C  boiling_margin_C  ranking_score
parallel_ribbed                  0.650              0.02           5.0          90.0      20.0                          48   16.0             2.0              0.75           81.3397        1.7420             8.7753                0.0011     157.8247          106.0753       158.6160
parallel_ribbed                  0.650              0.02           5.0          90.0      20.0                          48   16.0             2.0              1.00           81.3397        1.7420             9.3943                0.0011     157.8247          106.0753       158.5661
parallel_ribbed                  0.650              0.03           5.0          90.0      20.0                   